# Extended Data Fig. 3c: gene sets

Twenty genes for each set, chosen by rules fixed before the sets were scored and using no data from the section scored in the figure.
Absent cell types: the first 20 mouse canonical PanglaoDB markers present in the data. Excitatory neurons (positive control): the mouse
cortex reference of Tasic et al. 2018, ranked by detection in glutamatergic cells minus detection in all other cells. The reference covers
cortex only, so this set marks forebrain excitatory neurons. Writes `data/F1/ED3c_gene_sets.json`, read by `REV_GSD_base_brain` and
`ExtendedDataFig3_ExtendedDataFig4a-b`.


In [1]:
import json
import os

import decoupler as dc
import numpy as np
import pandas as pd
import scanpy as sc
import squidpy as sq


/home/cenk/miniconda3/envs/spatial_analysis/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


/home/cenk/miniconda3/envs/spatial_analysis/lib/python3.11/site-packages/docrep/decorators.py:43: SyntaxWarning: 'n_jobs' is not a valid key!
  doc = func(self, args[0].__doc__, *args[1:], **kwargs)
/home/cenk/miniconda3/envs/spatial_analysis/lib/python3.11/site-packages/docrep/decorators.py:43: SyntaxWarning: 'show_progress_bar' is not a valid key!
  doc = func(self, args[0].__doc__, *args[1:], **kwargs)


In [2]:
# set working directory
# repository root (the folder containing `notebooks/`); safe to re-run this cell
import sys
project_dir = os.getcwd()
if os.path.basename(project_dir) == "notebooks":
    project_dir = os.path.dirname(project_dir)
project_dir += "/"
sys.path.insert(0, project_dir + "notebooks")  # helpers.py and score_signatures.py
working_dir = project_dir
os.chdir(working_dir)

# genes of the section scored in Extended Data Fig. 3c
var = pd.Index(sc.read_h5ad(project_dir + "processed_data/visium_brain_adata.h5ad", backed="r").var_names)


In [3]:
# 1. Absent cell types (14_unit_tests cells 10-11): the first 20 mouse canonical markers in the data, without the sensitivity filter
markers = dc.op.resource("PanglaoDB", organism="mouse")
markers = markers[markers["mouse"].astype(bool) & markers["canonical_marker"].astype(bool)]
markers = markers[~markers.duplicated(["cell_type", "genesymbol"])]

gene_sets = {}
for ct in ["Cardiomyocytes", "Embryonic stem cells", "Hepatocytes"]:
    gene_sets[ct] = [gene for gene in markers.loc[markers["cell_type"] == ct, "genesymbol"] if gene in var][:20]


In [4]:
# 2. Excitatory neurons: top 20 genes in the data by detection in glutamatergic cells minus detection in all other cells
reference = sq.datasets.sc_mouse_cortex(path=os.path.expanduser("~/.cache/squidpy/sc_mouse_cortex.h5ad"))
glutamatergic = (reference.obs["cell_class"] == "Glutamatergic").values
print(reference.obs["cell_class"].value_counts().to_string())


def detection(x):
    return np.asarray((x > 0).mean(axis=0)).ravel()


ranking = pd.DataFrame({
    "gene": reference.var_names,
    "detection, glutamatergic": detection(reference.X[glutamatergic]),
    "detection, other cells": detection(reference.X[~glutamatergic]),
})
ranking["difference"] = ranking["detection, glutamatergic"] - ranking["detection, other cells"]
ranking = ranking[ranking["gene"].isin(var)].sort_values(["difference", "gene"], ascending=[False, True])
gene_sets["Excitatory neurons"] = ranking["gene"].head(20).tolist()
print(ranking.head(20).round(3).to_string(index=False))


cell_class
Glutamatergic    10952
GABAergic         9464
Non-Neuronal       992
Endothelial        289


   gene  detection, glutamatergic  detection, other cells  difference
Neurod2                     0.984                   0.014       0.970
   Tbr1                     0.971                   0.029       0.942
Gm11549                     0.984                   0.067       0.917
  Satb2                     0.939                   0.026       0.913
   Sv2b                     0.996                   0.100       0.895
   Nrn1                     0.984                   0.101       0.883
Neurod6                     0.948                   0.068       0.881
Slc17a7                     0.999                   0.122       0.877
  Vsig2                     0.949                   0.091       0.857
  Itpka                     0.966                   0.140       0.826
   Egr3                     0.911                   0.096       0.816
  Rprml                     0.996                   0.191       0.806
  Kcnv1                     0.938                   0.139       0.798
   Cobl             

In [5]:
# 3. Save
with open(working_dir + "data/F1/ED3c_gene_sets.json", "w") as f:
    json.dump(gene_sets, f, indent=1)
for ct, genes in gene_sets.items():
    print(f"{ct} ({len(genes)}): {', '.join(genes)}")


Cardiomyocytes (20): Acta1, Actc1, Actn2, Adipor1, Adipor2, Adm, Alpk3, Ankrd1, Aqp1, Art3, Asb2, Atp2a2, Bdnf, Bmp4, Casq2, Cd36, Ckb, Ckm, Ckmt2, Cpt1a
Embryonic stem cells (20): Cd9, Cdh1, Ctnnb1, Dnmt3b, Epcam, Esrrb, Fbxo15, Fut4, Gjb1, Gjc1, Hes1, Hhex, Il6st, Itga6, Itgb1, Kit, Klf4, L1td1, Lef1, Lifr
Hepatocytes (20): Abcd3, Acadm, Acat3, Acat2, Acly, Acss2, Afp, Ahr, Aldh6a1, Ambp, Ang, Apoa1, Apoa2, Apoc3, Apoh, Apom, Aqp9, Ar, Arg1, Asgr1
Excitatory neurons (20): Neurod2, Tbr1, Gm11549, Satb2, Sv2b, Nrn1, Neurod6, Slc17a7, Vsig2, Itpka, Egr3, Rprml, Kcnv1, Cobl, Emx1, Fam212b, Klf10, Nptx1, Rtn4r, Arc
